# 04 — Model Training

This notebook trains both tiers of the multilingual spam classifier:

**Tier 1 — MuRIL Transformer** (`google/muril-base-cased`)
- Fine-tuned on the combined multilingual training split
- Weighted cross-entropy loss (no oversampling)
- AdamW + linear warmup, early stopping on macro F1

**Tier 2 — LightGBM Fallback**
- TF-IDF char n-grams (2–5) + engineered features
- SMOTE for class balance
- Activated when MuRIL confidence < 0.75 or language is unknown

> **Prerequisites**: Run notebooks 01, 02, and 03 first.

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))

import pandas as pd
import numpy as np
import torch
from pathlib import Path

from src.model import train_muril, train_fallback, MURIL_SAVE_DIR, MODEL_DIR

DATA_DIR  = Path('../data')
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(exist_ok=True)

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Libraries loaded.')

CUDA: False
Libraries loaded.


## 1. Load Processed Splits

In [2]:
df_train = pd.read_csv(DATA_DIR / 'train.csv')
df_val   = pd.read_csv(DATA_DIR / 'val.csv')
df_test  = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}')
print('\nTrain label distribution:')
print(df_train.groupby(['language', 'label']).size().unstack(fill_value=0))

Train: 3,617 | Val: 776 | Test: 776

Train label distribution:
label        0    1
language           
en        3160  457


## 2. Tier 1: MuRIL Fine-tuning

### Why MuRIL?
- `google/muril-base-cased` was pre-trained on 17 Indian languages including **Assamese**.
- `ai4bharat/indic-bert` does NOT include Assamese in its pre-training data.
- MuRIL natively supports all 4 target languages: English, Hindi, Bengali, Assamese.

This decision is documented in `src/model.py`.

In [3]:
# Training takes ~20 min on GPU / ~2–3 hr on CPU for full multilingual dataset
# Reduce max_epochs for a quick smoke test

train_muril(
    df_train=df_train,
    df_val=df_val,
    max_epochs=5,
    batch_size=32,
    warmup_ratio=0.10,
    patience=2,
)

print(f'MuRIL saved to: {MURIL_SAVE_DIR}')

[muril] Loading tokeniser: google/muril-base-cased


Map:   0%|          | 0/3617 [00:00<?, ? examples/s]

Map:   0%|          | 0/776 [00:00<?, ? examples/s]

[muril] Class weights: {0: 0.5723101265822785, 1: 3.9573304157549236}
[muril] Loading model: google/muril-base-cased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

C:\Users\nj465\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[muril] Model saved → c:\Users\nj465\Downloads\Spam SMS Classification\Spam SMS Classification\models\muril_finetuned
MuRIL saved to: c:\Users\nj465\Downloads\Spam SMS Classification\Spam SMS Classification\models\muril_finetuned


## 3. Tier 2: LightGBM Fallback

In [7]:
# Ensure text_clean exists
for df_ in [df_train, df_val]:
    if 'text_clean' not in df_.columns:
        df_['text_clean'] = df_['text'].fillna('')

train_fallback(
    df_train=df_train,
    df_val=df_val,
    ngram_range=(2, 5),
    max_tfidf_features=30_000,
)

print('LightGBM fallback training complete.')

[fallback] Building TF-IDF features (char n-grams)...
[fallback] Applying SMOTE for class balance...
[fallback] Training LightGBM classifier...
[LightGBM] [Info] Number of positive: 3160, number of negative: 3160
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.092012 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 329970
[LightGBM] [Info] Number of data points in the train set: 6320, number of used features: 7045
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.0694381
Early stopping, best iteration is:
[110]	valid_0's binary_logloss: 0.0689983
[fallback] Saved TF-IDF → c:\Users\nj465\Downloads\Spam SMS Classification\Spam SMS Classificatio

## 4. Quick Validation Check (Tier 1)

In [8]:
from transformers import pipeline as hf_pipeline
from sklearn.metrics import f1_score, classification_report

pipe = hf_pipeline(
    'text-classification',
    model=str(MURIL_SAVE_DIR),
    tokenizer=str(MURIL_SAVE_DIR),
    device=0 if torch.cuda.is_available() else -1,
    truncation=True, max_length=128,
)

val_texts = df_val['text'].tolist()[:200]   # quick check on first 200
val_labels = df_val['label'].tolist()[:200]

results = pipe(val_texts)
preds = [int(r['label'].split('_')[1]) for r in results]

macro_f1 = f1_score(val_labels, preds, average='macro')
print(f'\nValidation Macro F1 (first 200 samples): {macro_f1:.4f}')
print(classification_report(val_labels, preds, target_names=['HAM', 'SPAM']))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Validation Macro F1 (first 200 samples): 0.9870
              precision    recall  f1-score   support

         HAM       1.00      0.99      1.00       179
        SPAM       0.95      1.00      0.98        21

    accuracy                           0.99       200
   macro avg       0.98      1.00      0.99       200
weighted avg       1.00      0.99      1.00       200



## 5. Quick Validation Check (Tier 2 — Fallback)

In [6]:
from src.model import predict_fallback

val_texts_clean = df_val['text_clean'].fillna('').tolist()[:200]
feat_df = df_val.iloc[:200]

labels_pred, confs_pred = predict_fallback(val_texts_clean, feat_df)

macro_f1_fb = f1_score(val_labels, labels_pred, average='macro')
print(f'Fallback Validation Macro F1 (first 200 samples): {macro_f1_fb:.4f}')
print(classification_report(val_labels, labels_pred, target_names=['HAM', 'SPAM']))

Fallback Validation Macro F1 (first 200 samples): 0.9468
              precision    recall  f1-score   support

         HAM       0.99      0.99      0.99       179
        SPAM       0.90      0.90      0.90        21

    accuracy                           0.98       200
   macro avg       0.95      0.95      0.95       200
weighted avg       0.98      0.98      0.98       200



C:\Users\nj465\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 6. Training Summary

| Component | Status |
|-----------|--------|
| MuRIL fine-tuned model | Saved to `models/muril_finetuned/` |
| LightGBM TF-IDF | Saved to `models/fallback_tfidf.pkl` |
| LightGBM classifier | Saved to `models/fallback_lgbm.pkl` |

**Next step**: Run `notebooks/05_evaluation.ipynb` for full test-set evaluation.